[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roberthsu2003/PythonForDataAnalysis/blob/master/%E8%A6%96%E8%A6%BA%E5%8C%96%E8%A1%A8%E6%A0%BC/%E5%AD%B8%E7%94%9F%E7%B4%A0%E6%9D%90/%E7%AF%84%E4%BE%8B_%E8%8C%B6%E8%AA%9E%E6%99%82%E5%85%89_%E6%9C%83%E7%99%BC%E4%BA%AE%E7%9A%84KPI%E9%80%B1%E5%A0%B1.ipynb)

# 🎨 茶語時光：做一份「會發亮」的 KPI 報表

> 🎯 **情境**：每季的營運會議，區經理都要盯著一張密密麻麻的數字表找問題。老闆說：「能不能讓沒達標的自動變紅、表現最好的自動亮起來？我要一眼就看到重點。」你決定用 pandas 的 Styler 幫表格「化妝」，最後匯出成帶顏色的 Excel。

| 素材檔案 | 內容 |
| --- | --- |
| `門市KPI_2024Q2.csv` | 第二季各門市的營收目標、實際營收、去年同期、來客數、評分、客訴、新會員 |
| `品項月銷量_2024.csv` | 10 個品項 × 12 個月的杯數 |

> ⚠️ 「茶語時光」是課程虛構的品牌，資料為模擬資料；節日與颱風假的日期參考台灣實際情況，方便你對照生活經驗。

## 🗺️ 今日任務地圖

| 關卡 | 要回答的問題 |
| --- | --- |
| ⬜ 任務 1 | 先算出關鍵指標，再幫數字「換衣服」 |
| ⬜ 任務 2 | 內建螢光筆——最好、最差、缺資料 |
| ⬜ 任務 3 | 自訂規則——沒達標變紅、客訴太多變紅字 |
| ⬜ 任務 4 | 熱力圖——一眼看出每個品項的旺季 |
| ⬜ 任務 5 | 匯出成「帶顏色」的 Excel，直接寄給區經理 |

**怎麼用這本 Notebook**：從上到下按 `Shift + Enter` 一格一格執行。每個任務都會先講「老闆想知道什麼」，再給程式，最後用 💡 **看懂結果** 告訴你這個數字代表什麼。

In [1]:
# 📦 準備素材：在 Colab 開啟時，自動下載本章素材壓縮檔並解壓縮
#    （若是在自己電腦解壓縮後開啟，檔案已經在旁邊，會直接使用）
import os, zipfile, urllib.request

FILES = [
    '門市KPI_2024Q2.csv',
    '品項月銷量_2024.csv',
]
ZIP_URL = 'https://github.com/roberthsu2003/PythonForDataAnalysis/raw/master/%E8%A6%96%E8%A6%BA%E5%8C%96%E8%A1%A8%E6%A0%BC/%E8%A6%96%E8%A6%BA%E5%8C%96%E8%A1%A8%E6%A0%BC_%E5%AD%B8%E7%94%9F%E7%B4%A0%E6%9D%90.zip'

if not all(os.path.exists(f) for f in FILES):
    urllib.request.urlretrieve(ZIP_URL, 'materials.zip')            # 下載本章素材壓縮檔
    with zipfile.ZipFile('materials.zip') as z:
        for name in z.namelist():
            inner = name.split('/', 1)[1] if '/' in name else name   # 去掉壓縮檔最外層的資料夾名稱
            if inner and not name.endswith('/'):
                os.makedirs(os.path.dirname(inner) or '.', exist_ok=True)
                with z.open(name) as src, open(inner, 'wb') as dst:
                    dst.write(src.read())

print('✅ 素材準備完成：', [f for f in FILES if os.path.exists(f)])

✅ 素材準備完成： ['門市KPI_2024Q2.csv', '品項月銷量_2024.csv']


## 任務 1：先算出關鍵指標，再幫數字「換衣服」

**白話說明**：Styler 只改「外觀」，不改資料。千分位、百分比、小數位數，都用 `format` 設定。

In [2]:
import pandas as pd

kpi = pd.read_csv('門市KPI_2024Q2.csv', index_col='門市')
kpi['達成率'] = kpi['實際營收'] / kpi['營收目標']                    # 實際 ÷ 目標
kpi['年成長率'] = kpi['實際營收'] / kpi['去年同期營收'] - 1          # 和去年同期比
kpi['客單價'] = kpi['實際營收'] / kpi['來客數']
display(kpi.round(3))                                                  # 素顏的表格：很難一眼看出重點

styled = kpi.style.format({
    '營收目標': '{:,.0f}', '實際營收': '{:,.0f}', '去年同期營收': '{:,.0f}', '來客數': '{:,}',
    '達成率': '{:.1%}', '年成長率': '{:+.1%}', '客單價': '{:.0f}', 'Google評分': '{:.1f}',
}, na_rep='尚無資料').set_caption('2024 第二季門市 KPI')
styled

,營收目標,實際營收,去年同期營收,來客數,Google評分,客訴數,新會員數,達成率,年成長率,客單價
門市,,,,,,,,,,
台北信義店,2270000,2203389,1912862,26276,4.5,3,256,0.971,0.152,83.856
台北西門店,2710000,2468698,2143567,27204,3.7,14,245,0.911,0.152,90.748
台中逢甲店,2620000,2676839,2396626,30578,4.2,6,183,1.022,0.117,87.541
台南成大店,1760000,1903302,1677748,21380,4.6,2,89,1.081,0.134,89.023
高雄巨蛋店,2200000,2155275,1872366,24015,NaN,9,111,0.980,0.151,89.747


,營收目標,實際營收,去年同期營收,來客數,Google評分,客訴數,新會員數,達成率,年成長率,客單價
門市,,,,,,,,,,
台北信義店,"2,270,000","2,203,389","1,912,862","26,276",4.5,3,256,97.1%,+15.2%,84
台北西門店,"2,710,000","2,468,698","2,143,567","27,204",3.7,14,245,91.1%,+15.2%,91
台中逢甲店,"2,620,000","2,676,839","2,396,626","30,578",4.2,6,183,102.2%,+11.7%,88
台南成大店,"1,760,000","1,903,302","1,677,748","21,380",4.6,2,89,108.1%,+13.4%,89
高雄巨蛋店,"2,200,000","2,155,275","1,872,366","24,015",尚無資料,9,111,98.0%,+15.1%,90


## 任務 2：內建螢光筆——最好、最差、缺資料

In [3]:
styled2 = (styled
           .highlight_max(subset=['達成率', '年成長率', 'Google評分', '新會員數'], props='background-color:#c6efce; font-weight:bold;')  # 每欄最好：綠底
           .highlight_min(subset=['達成率', '年成長率', 'Google評分'], props='background-color:#ffc7ce;')                               # 每欄最差：紅底
           .highlight_null(color='#ffeb9c'))                                                                                          # 缺資料：黃底
styled2

,營收目標,實際營收,去年同期營收,來客數,Google評分,客訴數,新會員數,達成率,年成長率,客單價
門市,,,,,,,,,,
台北信義店,"2,270,000","2,203,389","1,912,862","26,276",4.5,3,256,97.1%,+15.2%,84
台北西門店,"2,710,000","2,468,698","2,143,567","27,204",3.7,14,245,91.1%,+15.2%,91
台中逢甲店,"2,620,000","2,676,839","2,396,626","30,578",4.2,6,183,102.2%,+11.7%,88
台南成大店,"1,760,000","1,903,302","1,677,748","21,380",4.6,2,89,108.1%,+13.4%,89
高雄巨蛋店,"2,200,000","2,155,275","1,872,366","24,015",尚無資料,9,111,98.0%,+15.1%,90


## 任務 3：自訂規則——沒達標變紅、客訴太多變紅字

**公司的規則**：達成率 < 95% 紅底白字；≥ 105% 綠底；客訴 ≥ 10 件紅色粗體。

In [4]:
def rate_color(v):                                                    # 一次處理一格「達成率」
    if v < 0.95:
        return 'background-color:#c00000; color:white; font-weight:bold;'
    if v >= 1.05:
        return 'background-color:#00b050; color:white;'
    return None

def complaint_color(v):
    return 'color:#c00000; font-weight:bold;' if v >= 10 else None

styled3 = (styled
           .map(rate_color, subset=['達成率'])                         # 套用到「達成率」欄（舊版 pandas 寫作 applymap）
           .map(complaint_color, subset=['客訴數'])
           .bar(subset=['年成長率'], align=0, color=['#ff9999', '#99cc99'], vmin=-0.2, vmax=0.3))   # 成長率資料條：負紅正綠，0 對齊
styled3

,營收目標,實際營收,去年同期營收,來客數,Google評分,客訴數,新會員數,達成率,年成長率,客單價
門市,,,,,,,,,,
台北信義店,"2,270,000","2,203,389","1,912,862","26,276",4.5,3,256,97.1%,+15.2%,84
台北西門店,"2,710,000","2,468,698","2,143,567","27,204",3.7,14,245,91.1%,+15.2%,91
台中逢甲店,"2,620,000","2,676,839","2,396,626","30,578",4.2,6,183,102.2%,+11.7%,88
台南成大店,"1,760,000","1,903,302","1,677,748","21,380",4.6,2,89,108.1%,+13.4%,89
高雄巨蛋店,"2,200,000","2,155,275","1,872,366","24,015",尚無資料,9,111,98.0%,+15.1%,90


> 💡 **看懂結果**：西門店**達成率最低、客訴最多、評分也最低**——三個警訊同時亮起，下季要優先改善；台南成大店則是全面亮綠燈。
> 有了顏色，區經理不用一格一格看數字，**三秒鐘就找到問題門市**。
>
> 🤖 **換你問 AI**：「請用 pandas Styler 格式化這份 KPI 表：達成率顯示百分比，低於 95% 紅底白字、高於 105% 綠底；客訴 ≥ 10 紅字；成長率用資料條呈現。」

## 任務 4：熱力圖——一眼看出每個品項的旺季

**白話說明**：`background_gradient` 讓數字越大顏色越深，一張表就能看出「季節性」。`axis=1` 代表**每個品項自己比較**（每一列各自上色）。

In [5]:
heat = pd.read_csv('品項月銷量_2024.csv', index_col='品項')
heat_style = (heat.style
              .background_gradient(cmap='YlOrBr', axis=1)              # 每一列（每個品項）各自由淺到深
              .format('{:,}')
              .set_caption('2024 各品項每月杯數（顏色越深＝該品項的旺季）'))
heat_style

,1月,2月,3月,4月,5月,6月,7月,8月,9月,10月,11月,12月
品項,,,,,,,,,,,,
珍珠奶茶,"11,607","11,542","13,572","12,060","12,560","13,163","12,293","13,416","12,715","12,313","12,064","11,619"
四季春青茶,"7,373","7,197","8,481","7,624","8,002","8,358","7,674","8,252","8,148","7,762","7,475","7,298"
鮮奶綠,"6,178","5,994","6,986","6,457","6,725","7,168","6,475","7,182","6,711","6,635","6,340","6,051"
冬瓜檸檬,"5,502","5,419","6,337","5,722","5,898","6,185","5,867","6,264","6,103","5,824","5,584","5,404"
多多綠,"4,897","4,853","5,585","4,929","5,225","5,546","5,101","5,516","5,265","5,000","4,898","4,853"
黑糖珍珠鮮奶,"7,333","7,019","8,306","7,469","7,805","8,392","7,462","8,356","7,944","7,751","7,398","7,157"
芒果冰沙,252,409,"1,222","3,280","7,116","10,659","10,486","9,829","6,761","2,842",664,159
熱可可,"1,865",988,366,0,0,0,0,0,0,11,304,"1,255"
檸檬紅茶,"5,191","5,064","6,035","5,362","5,632","5,905","5,449","5,939","5,640","5,443","5,252","5,187"


> 💡 **看懂結果**：芒果冰沙的深色集中在 6～9 月，熱可可只有 1～2 月、12 月有顏色；芋頭鮮奶 1～3 月是空白（還沒上市）。**採購與行銷檔期就照著這張表排。**

## 任務 5：匯出成「帶顏色」的 Excel，直接寄給區經理

In [6]:
with pd.ExcelWriter('2024Q2_KPI報表.xlsx') as writer:
    styled3.to_excel(writer, sheet_name='門市KPI')                      # 顏色、粗體會一起保留
    heat_style.to_excel(writer, sheet_name='品項季節性')
print('✅ 已輸出 2024Q2_KPI報表.xlsx（用 Excel 打開就能看到顏色）')
print('原始資料沒有被改變：實際營收合計 =', f"{kpi['實際營收'].sum():,}")

✅ 已輸出 2024Q2_KPI報表.xlsx（用 Excel 打開就能看到顏色）
原始資料沒有被改變：實際營收合計 = 11,407,503


---

## ✍️ 換你動手：練習區

每一題都有三格：**① 半成品程式**（把 `___` 換成正確的程式）→ **② ✅ 檢查格**（自動告訴你對不對）→ **③ 🔑 參考解答**（折疊起來，先自己試）。

> 看到 `NameError: name '___' is not defined` 不用緊張，那代表還有空格沒填。

In [7]:
# @title 🧰 練習檢查小幫手（執行一次就好，不用看懂裡面的程式）
import math
import numpy as np
import pandas as pd


def _same(ans, expected, ordered=False):
    if isinstance(expected, (pd.DataFrame, pd.Series)):
        if type(ans) is not type(expected):
            return False
        try:
            assert_equal = pd.testing.assert_frame_equal if isinstance(expected, pd.DataFrame) else pd.testing.assert_series_equal
            if not ordered:
                ans, expected = ans.sort_index(), expected.sort_index()
            assert_equal(ans, expected, check_names=False, check_dtype=False,
                         check_index_type=False, check_exact=False, rtol=1e-3, atol=0.06,
                         **({'check_column_type': False} if isinstance(expected, pd.DataFrame) else {}))
            return True
        except (AssertionError, TypeError, ValueError):
            return False
    if isinstance(expected, np.ndarray):
        return np.asarray(ans).shape == expected.shape and np.allclose(np.asarray(ans, dtype=float), expected, atol=0.06)
    if isinstance(expected, (int, float, np.number)) and not isinstance(expected, bool):
        try:
            return math.isclose(float(ans), float(expected), rel_tol=1e-3, abs_tol=0.06)
        except (TypeError, ValueError):
            return False
    if isinstance(expected, (list, tuple, set)):
        return list(ans) == list(expected) if not isinstance(expected, set) else set(ans) == expected
    return ans == expected


def check(name, expected, ordered=False):
    """比對學生的變數 name 和正確答案 expected，印出 ✅ 或 ❌；ordered=True 時連排列順序也要一樣。"""
    if name not in globals():
        print(f'⏳ 還找不到變數 {name}：請先把上一格的 ___ 填好並執行')
        return
    ans = globals()[name]
    if _same(ans, expected, ordered):
        print(f'✅ {name} 答對了！')
        return
    print(f'❌ {name} 還不對，再檢查看看：')
    if type(ans) is not type(expected):
        print(f'   型態不同：你的是 {type(ans).__name__}，應該是 {type(expected).__name__}')
    elif hasattr(expected, 'shape') and getattr(ans, 'shape', None) != expected.shape:
        print(f'   大小不同：你的是 {ans.shape}，應該是 {expected.shape}')
    else:
        print('   型態和大小都對了，是數值' + ('或排列順序' if ordered else '') + '不一樣——檢查一下篩選條件或計算方式')


print('🧰 檢查小幫手準備好了：每題寫完後，執行下方的「✅ 檢查」格')

🧰 檢查小幫手準備好了：每題寫完後，執行下方的「✅ 檢查」格


### ✍️ 練習 1：「客單價」高於全公司平均的格子，標成淡藍底

<details><summary>💡 卡住了？點我看提示</summary>

`styled.map(函式, subset=['客單價'])` 會把每一格的值丟進函式；函式回傳 CSS 字串（例如 `'background-color:#ddebf7;'`）就會上色，回傳 `None` 就不變。

</details>

In [ ]:
# ✏️ 練習 1：把 ___ 換成正確的程式，再按執行
avg = kpi['客單價'].___()
ex1 = styled.map(lambda v: 'background-color:#ddebf7;' if v > ___ else None, subset=['客單價'])
ex1

In [ ]:
# @title ✅ 執行這格，檢查練習 1
check('avg', kpi['客單價'].mean())

<details><summary>🔑 先自己試試看！真的想不出來再點開參考解答</summary>

```python
avg = kpi['客單價'].mean()
ex1 = styled.map(lambda v: 'background-color:#ddebf7;' if v > avg else None, subset=['客單價'])
ex1
```

</details>

### ✍️ 練習 2：給老闆看的版本：隱藏「去年同期營收」與「營收目標」兩欄

<details><summary>💡 卡住了？點我看提示</summary>

`.hide(subset=[欄位, 欄位], axis='columns')`，只是「不顯示」，資料本身還在。

</details>

In [ ]:
# ✏️ 練習 2：把 ___ 換成正確的程式，再按執行
ex2 = styled.hide(subset=[___, ___], axis='columns')
ex2

<details><summary>🔑 先自己試試看！真的想不出來再點開參考解答</summary>

```python
ex2 = styled.hide(subset=['去年同期營收', '營收目標'], axis='columns')
ex2
```

</details>

### ✍️ 練習 3：熱力圖改成 `axis=0`：顏色的意思變成什麼？

<details><summary>💡 卡住了？點我看提示</summary>

`axis=1` 是「每一列（品項）自己比」；`axis=0` 是「每一欄（月份）自己比」——看的是「這個月誰賣最好」。

</details>

In [ ]:
# ✏️ 練習 3：把 ___ 換成正確的程式，再按執行
ex3 = heat.style.background_gradient(cmap='YlOrBr', axis=___).format('{:,}')
ex3

<details><summary>🔑 先自己試試看！真的想不出來再點開參考解答</summary>

```python
ex3 = heat.style.background_gradient(cmap='YlOrBr', axis=0).format('{:,}')
ex3
# axis=0 → 每個月份（每一欄）各自上色，看的是「這個月誰賣最好」，而不是「這個品項哪個月最好」
```

</details>

---

## 🏆 任務完成！今天你用程式做到了這些事

- ✅ 任務 1：先算出關鍵指標，再幫數字「換衣服」
- ✅ 任務 2：內建螢光筆——最好、最差、缺資料
- ✅ 任務 3：自訂規則——沒達標變紅、客訴太多變紅字
- ✅ 任務 4：熱力圖——一眼看出每個品項的旺季
- ✅ 任務 5：匯出成「帶顏色」的 Excel，直接寄給區經理

> 🎯 **帶走一句話**：程式只是工具，真正有價值的是你從資料裡找到的那句「所以我們應該……」。
> 把上面每個 💡 **看懂結果** 用自己的話寫成一句建議，就是一份可以交給老闆的報告。